# 03 — Velocity analysis

This notebook analyzes frame-to-frame displacement and velocity distributions from preprocessed TrackMate trajectories.

Input generated by `01_trackmate_preprocessing.ipynb`:

```text
results/preprocessed/filtered_spots_combined.csv
```

The notebook performs:
- frame-to-frame link extraction,
- displacement and velocity calculation,
- velocity distribution visualization,
- Rayleigh reference curve fitting,
- high-velocity fraction estimation,
- per-cell velocity summary generation.


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


plt.style.use("default")

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RESULTS_DIR = PROJECT_DIR / "results"
PREPROCESSED_DIR = RESULTS_DIR / "preprocessed"
VELOCITY_DIR = RESULTS_DIR / "velocity"

FIGURES_DIR = PROJECT_DIR / "figures"
VELOCITY_FIGURES_DIR = FIGURES_DIR / "velocity"

for directory in [
    RESULTS_DIR,
    PREPROCESSED_DIR,
    VELOCITY_DIR,
    FIGURES_DIR,
    VELOCITY_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


FILTERED_SPOTS_PATH = PREPROCESSED_DIR / "filtered_spots_combined.csv"

CONDITION_ORDER = [
    "control",
    "500uM_H2O2",
    "500uM_H2O2_30min",
]

CONDITION_LABELS = {
    "control": "Control",
    "500uM_H2O2": "500 µM H₂O₂",
    "500uM_H2O2_30min": "500 µM H₂O₂ 30 min",
}

VELOCITY_THRESHOLD_PERCENTILE = 95
N_BINS = 60

print("Project directory:", PROJECT_DIR)
print("Input file:", FILTERED_SPOTS_PATH)
print("Configuration loaded")


## 2. Load preprocessed trajectories

In [ ]:
if not FILTERED_SPOTS_PATH.exists():
    raise FileNotFoundError(
        f"Preprocessed file was not found: {FILTERED_SPOTS_PATH}. "
        "Run 01_trackmate_preprocessing.ipynb first."
    )

spots = pd.read_csv(FILTERED_SPOTS_PATH)

required_columns = [
    "protein",
    "condition",
    "sample",
    "cell",
    "file_name",
    "track_id",
    "x",
    "y",
    "t",
    "frame",
]

missing_columns = [
    column for column in required_columns
    if column not in spots.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(spots.columns)}"
    )

spots = spots.sort_values(
    ["protein", "sample", "cell", "condition", "track_id", "frame"]
).reset_index(drop=True)

print("Loaded spots:", len(spots))
print("Number of tracks:", spots["track_id"].nunique())

spots.head()


## 3. Link extraction and velocity calculation

In [ ]:
def calculate_frame_to_frame_links(spots_table):
    """Calculate frame-to-frame displacement and velocity for all trajectories."""
    link_rows = []

    group_columns = [
        "protein",
        "condition",
        "sample",
        "cell",
        "file_name",
        "track_id",
    ]

    for keys, track in spots_table.groupby(group_columns, sort=False):
        protein, condition, sample, cell, file_name, track_id = keys
        track = track.sort_values("frame")

        if len(track) < 2:
            continue

        x = track["x"].to_numpy(dtype=float)
        y = track["y"].to_numpy(dtype=float)
        t = track["t"].to_numpy(dtype=float)
        frames = track["frame"].to_numpy(dtype=int)

        dx = np.diff(x)
        dy = np.diff(y)
        dt = np.diff(t)
        frame_step = np.diff(frames)

        displacement_um = np.sqrt(dx**2 + dy**2)

        with np.errstate(divide="ignore", invalid="ignore"):
            velocity_um_s = displacement_um / dt

        for index in range(len(displacement_um)):
            if not np.isfinite(velocity_um_s[index]) or velocity_um_s[index] <= 0:
                continue

            link_rows.append({
                "protein": protein,
                "condition": condition,
                "sample": sample,
                "cell": cell,
                "file_name": file_name,
                "track_id": track_id,
                "start_frame": frames[index],
                "end_frame": frames[index + 1],
                "frame_step": frame_step[index],
                "dt_s": dt[index],
                "dx_um": dx[index],
                "dy_um": dy[index],
                "displacement_um": displacement_um[index],
                "velocity_um_s": velocity_um_s[index],
            })

    return pd.DataFrame(link_rows)


links = calculate_frame_to_frame_links(spots)

links_path = VELOCITY_DIR / "frame_to_frame_links.csv"
links.to_csv(links_path, index=False)

print("Number of links:", len(links))
print("Saved:", links_path)

links.head()


## 4. Rayleigh reference distribution

For ideal isotropic two-dimensional Brownian diffusion, x and y displacements are normally distributed. The radial displacement or velocity magnitude follows a Rayleigh distribution.

The Rayleigh scale parameter is estimated as:

```text
sigma = sqrt(mean(v^2) / 2)
```


In [ ]:
def estimate_rayleigh_sigma(values):
    """Estimate Rayleigh scale parameter from positive values."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]

    if len(values) == 0:
        return np.nan

    return np.sqrt(np.mean(values**2) / 2)


def rayleigh_pdf(x, sigma):
    """Rayleigh probability density function without scipy dependency."""
    x = np.asarray(x, dtype=float)

    if not np.isfinite(sigma) or sigma <= 0:
        return np.full_like(x, np.nan)

    return (x / sigma**2) * np.exp(-(x**2) / (2 * sigma**2))


def rayleigh_percentile(sigma, percentile):
    """Rayleigh percentile without scipy dependency."""
    probability = percentile / 100

    if not np.isfinite(sigma) or sigma <= 0:
        return np.nan

    return sigma * np.sqrt(-2 * np.log(1 - probability))


## 5. Control-based high-velocity thresholds

In [ ]:
threshold_rows = []

for protein in sorted(links["protein"].dropna().unique()):
    control_values = links.loc[
        (links["protein"] == protein)
        & (links["condition"] == "control"),
        "velocity_um_s",
    ].dropna().to_numpy()

    control_values = control_values[
        np.isfinite(control_values)
        & (control_values > 0)
    ]

    if len(control_values) == 0:
        continue

    empirical_threshold = np.percentile(
        control_values,
        VELOCITY_THRESHOLD_PERCENTILE,
    )

    sigma = estimate_rayleigh_sigma(control_values)
    rayleigh_threshold = rayleigh_percentile(
        sigma,
        VELOCITY_THRESHOLD_PERCENTILE,
    )

    threshold_rows.append({
        "protein": protein,
        "threshold_percentile": VELOCITY_THRESHOLD_PERCENTILE,
        "empirical_threshold_um_s": empirical_threshold,
        "rayleigh_sigma_um_s": sigma,
        "rayleigh_threshold_um_s": rayleigh_threshold,
        "n_control_links": len(control_values),
    })

velocity_thresholds = pd.DataFrame(threshold_rows)

thresholds_path = VELOCITY_DIR / "velocity_thresholds.csv"
velocity_thresholds.to_csv(thresholds_path, index=False)

print("Saved:", thresholds_path)
velocity_thresholds


## 6. Per-cell velocity summary

In [ ]:
velocity_summary_rows = []

for keys, group in links.groupby(
    ["protein", "condition", "sample", "cell", "file_name"],
    observed=True,
):
    protein, condition, sample, cell, file_name = keys

    threshold_row = velocity_thresholds[
        velocity_thresholds["protein"] == protein
    ]

    if threshold_row.empty:
        empirical_threshold = np.nan
        rayleigh_threshold = np.nan
    else:
        empirical_threshold = threshold_row["empirical_threshold_um_s"].iloc[0]
        rayleigh_threshold = threshold_row["rayleigh_threshold_um_s"].iloc[0]

    velocities = group["velocity_um_s"].dropna().to_numpy()
    displacements = group["displacement_um"].dropna().to_numpy()

    velocity_summary_rows.append({
        "protein": protein,
        "condition": condition,
        "sample": sample,
        "cell": cell,
        "file_name": file_name,
        "n_links": len(group),
        "median_displacement_um": np.median(displacements),
        "mean_displacement_um": np.mean(displacements),
        "median_velocity_um_s": np.median(velocities),
        "mean_velocity_um_s": np.mean(velocities),
        "velocity_threshold_um_s": empirical_threshold,
        "rayleigh_threshold_um_s": rayleigh_threshold,
        "high_velocity_fraction": np.mean(velocities > empirical_threshold)
        if np.isfinite(empirical_threshold)
        else np.nan,
        "high_velocity_percent": 100 * np.mean(velocities > empirical_threshold)
        if np.isfinite(empirical_threshold)
        else np.nan,
    })

velocity_summary = pd.DataFrame(velocity_summary_rows)

velocity_summary_path = VELOCITY_DIR / "cell_level_velocity_summary.csv"
velocity_summary.to_csv(velocity_summary_path, index=False)

print("Saved:", velocity_summary_path)

velocity_summary.sort_values(
    ["protein", "sample", "cell", "condition"],
    na_position="last",
)


## 7. Plotting functions

In [ ]:
def setup_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)


def get_available_conditions(data):
    return [
        condition for condition in CONDITION_ORDER
        if condition in data["condition"].unique()
    ]


def plot_velocity_distributions_with_rayleigh(protein):
    data = links[links["protein"] == protein].copy()

    if data.empty:
        print(f"No velocity data for {protein}")
        return

    conditions = get_available_conditions(data)
    n_conditions = len(conditions)

    if n_conditions == 0:
        return

    fig, axes = plt.subplots(
        1,
        n_conditions,
        figsize=(5 * n_conditions, 4),
        sharey=True,
    )

    if n_conditions == 1:
        axes = [axes]

    threshold_row = velocity_thresholds[
        velocity_thresholds["protein"] == protein
    ]

    threshold = (
        threshold_row["empirical_threshold_um_s"].iloc[0]
        if not threshold_row.empty
        else np.nan
    )

    max_velocity = np.percentile(data["velocity_um_s"].dropna(), 99.5)

    for ax, condition in zip(axes, conditions):
        values = data.loc[
            data["condition"] == condition,
            "velocity_um_s",
        ].dropna().to_numpy()

        values = values[np.isfinite(values) & (values > 0)]

        if len(values) == 0:
            continue

        ax.hist(
            values,
            bins=N_BINS,
            density=True,
            alpha=0.75,
            label="Data",
        )

        sigma = estimate_rayleigh_sigma(values)
        x = np.linspace(0, max_velocity, 400)
        y = rayleigh_pdf(x, sigma)

        ax.plot(
            x,
            y,
            linewidth=2.5,
            label="Rayleigh reference",
        )

        if np.isfinite(threshold):
            ax.axvline(
                threshold,
                linestyle="--",
                linewidth=2,
                label=f"{VELOCITY_THRESHOLD_PERCENTILE}% control threshold",
            )

            ax.text(
                threshold,
                ax.get_ylim()[1] * 0.85,
                f"{threshold:.1f} µm/s",
                rotation=90,
                va="top",
                ha="right",
            )

        ax.set_title(CONDITION_LABELS[condition])
        ax.set_xlabel("Velocity, µm/s")
        setup_axis(ax)

    axes[0].set_ylabel("Probability density")

    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        bbox_to_anchor=(1.02, 0.95),
        loc="upper left",
        frameon=False,
    )

    fig.suptitle(f"{protein}: frame-to-frame velocity with Rayleigh reference")
    plt.tight_layout()

    output_path = VELOCITY_FIGURES_DIR / f"{protein}_velocity_rayleigh_reference.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_displacement_distributions(protein):
    data = links[links["protein"] == protein].copy()

    if data.empty:
        print(f"No displacement data for {protein}")
        return

    conditions = get_available_conditions(data)
    n_conditions = len(conditions)

    fig, axes = plt.subplots(
        1,
        n_conditions,
        figsize=(5 * n_conditions, 4),
        sharey=True,
    )

    if n_conditions == 1:
        axes = [axes]

    max_displacement = np.percentile(data["displacement_um"].dropna(), 99.5)

    for ax, condition in zip(axes, conditions):
        values = data.loc[
            data["condition"] == condition,
            "displacement_um",
        ].dropna().to_numpy()

        values = values[np.isfinite(values) & (values >= 0)]

        ax.hist(
            values,
            bins=N_BINS,
            range=(0, max_displacement),
            density=True,
            alpha=0.75,
        )

        ax.set_title(CONDITION_LABELS[condition])
        ax.set_xlabel("Frame-to-frame displacement, µm")
        setup_axis(ax)

    axes[0].set_ylabel("Probability density")

    fig.suptitle(f"{protein}: frame-to-frame displacement")
    plt.tight_layout()

    output_path = VELOCITY_FIGURES_DIR / f"{protein}_displacement_distribution.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_high_velocity_fraction(protein):
    data = velocity_summary[
        velocity_summary["protein"] == protein
    ].copy()

    if data.empty:
        print(f"No velocity summary for {protein}")
        return

    data["condition"] = pd.Categorical(
        data["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    data = data.sort_values(["sample", "cell", "condition"])
    x_map = {condition: i for i, condition in enumerate(CONDITION_ORDER)}

    fig, ax = plt.subplots(figsize=(7, 5))

    for (sample, cell), group in data.groupby(["sample", "cell"], observed=True):
        group = group.sort_values("condition")

        x = group["condition"].map(x_map).astype(float).to_numpy()
        y = group["high_velocity_percent"].to_numpy()

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            alpha=0.55,
            label=f"S{sample} cell{cell}",
        )

    median_summary = (
        data.groupby("condition", observed=True)["high_velocity_percent"]
        .median()
        .reindex(CONDITION_ORDER)
    )

    ax.plot(
        range(len(CONDITION_ORDER)),
        median_summary.values,
        marker="o",
        linewidth=4,
        color="black",
        label="group median",
    )

    ax.set_xticks(range(len(CONDITION_ORDER)))
    ax.set_xticklabels(
        [CONDITION_LABELS[c] for c in CONDITION_ORDER],
        rotation=20,
    )

    ax.set_ylabel("High-velocity links per cell, %")
    ax.set_title(
        f"{protein}: high-velocity fraction per cell\n"
        f"threshold = {VELOCITY_THRESHOLD_PERCENTILE}th percentile of control"
    )

    setup_axis(ax)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
    plt.tight_layout()

    output_path = VELOCITY_FIGURES_DIR / f"{protein}_high_velocity_fraction.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


## 8. Generate velocity figures

In [ ]:
proteins = sorted(links["protein"].dropna().unique())

for protein in proteins:
    plot_displacement_distributions(protein)
    plot_velocity_distributions_with_rayleigh(protein)
    plot_high_velocity_fraction(protein)


## 9. Quick interpretation guide

- `frame_to_frame_links.csv` contains individual frame-to-frame displacement and velocity values.
- `velocity_thresholds.csv` contains control-based high-velocity thresholds for each protein.
- `cell_level_velocity_summary.csv` is the recommended table for condition-level comparison because the cell is the biological/statistical unit.
- Rayleigh curves represent an idealized reference for two-dimensional Brownian motion. Deviations from this reference may indicate heterogeneous diffusion, confinement, active transport, tracking artifacts, or mixed mobility states.


## 10. Output files

This notebook generates:

```text
results/velocity/frame_to_frame_links.csv
results/velocity/velocity_thresholds.csv
results/velocity/cell_level_velocity_summary.csv
figures/velocity/*_displacement_distribution.png
figures/velocity/*_velocity_rayleigh_reference.png
figures/velocity/*_high_velocity_fraction.png
```

These outputs are used by downstream notebooks for mobility-state analysis, HMM inference, sensitivity analysis, and final figure generation.
